In [ ]:
from pynq.overlays.base import BaseOverlay
import time
from datetime import datetime
base = BaseOverlay("base.bit")

In [ ]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
unsigned int write_gpio(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

//Function to read the value of a selected pin of PMODB
unsigned int read_gpio(unsigned int pin){
    gpio pin_in = gpio_open(pin);
    gpio_set_direction(pin_in, GPIO_IN);
    return gpio_read(pin_in);
}

/*
//void close_gpio(unsigned int pin){
unsigned int close_gpio(unsigned int pin){
    gpio gpio_pin = gpio_open(pin);
    gpio_close(gpio_pin);
    return 0;
}

//Function to reset all GPIO pins on the chosen PMOD
unsigned int reset_all_gpio(){
    for(unsigned int i = 0; i<= 7; ++i){
        close_gpio(i);
    }
    return 0;
}
*/

In [ ]:
#to reset all GPIOs of PMODB

#sequential approach
'''
write_gpio(0, 0)
write_gpio(1, 0)
write_gpio(2, 0)
write_gpio(3, 0)
write_gpio(4, 0)
write_gpio(5, 0)
write_gpio(6, 0)
write_gpio(7, 0)
'''

for pinId in range(0, 8):
    print("clearing PIN %d" % (pinId))
    write_gpio(pinId, 0)

#reset_all_gpio() #doesn't work

In [ ]:
btns = base.btns_gpio

GREEN_PIN_ID = 5
RED_PIN_ID = 6
BLUE_PIN_ID = 7

LED_ON = 1
LED_OFF = 0

def gpioLedOnOff(onTime, offTime):
    while True:
        write_gpio(GREEN_PIN_ID, LED_ON)
        write_gpio(RED_PIN_ID, LED_ON)
        write_gpio(BLUE_PIN_ID, LED_ON)
        time.sleep(onTime)

        write_gpio(GREEN_PIN_ID, LED_OFF)
        write_gpio(RED_PIN_ID, LED_OFF)
        write_gpio(BLUE_PIN_ID, LED_OFF)
        time.sleep(offTime)

        if btns.read() != 0:
            break
    return

def offAllLeds():
    write_gpio(GREEN_PIN_ID, LED_OFF)
    write_gpio(RED_PIN_ID, LED_OFF)
    write_gpio(BLUE_PIN_ID, LED_OFF)
    return

print("press any button to try different duty cycle or frequency")

frequency = 10 #in Hz
dutyCycle = 0.25 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
oneCycleLen=1/frequency

onTime = dutyCycle*oneCycleLen
offTime = (1-dutyCycle)*oneCycleLen

print("frequency: %d Hz onTime: %f offTime: %f Sec oneCycleDuration:%f Sec" % (frequency,onTime,offTime,oneCycleLen))

gpioLedOnOff(onTime, offTime)

'''
while True:
    write_gpio(5, 1)
    write_gpio(6, 1)
    write_gpio(7, 1)
    time.sleep(onTime)
    write_gpio(5, 0)
    write_gpio(6, 0)
    write_gpio(7, 0)
    time.sleep(offTime)
    if btns.read() != 0:
        break

write_gpio(5, 0)
write_gpio(6, 0)
write_gpio(7, 0)
'''


#experiments with different frequency and duty cycle

frequency = 100 #in Hz
dutyCycle = 0.25 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
oneCycleLen=1/frequency

onTime = dutyCycle*oneCycleLen
offTime = (1-dutyCycle)*oneCycleLen

print("frequency: %d Hz onTime: %f offTime: %f Sec oneCycleDuration:%f Sec" % (frequency,onTime,offTime,oneCycleLen))

#At this ON/OFF rate, LED doesn't appear to be flickering at all

gpioLedOnOff(onTime, offTime)
'''
while True:
    write_gpio(5, 1)
    write_gpio(6, 1)
    write_gpio(7, 1)
    time.sleep(onTime)
    write_gpio(5, 0)
    write_gpio(6, 0)
    write_gpio(7, 0)
    time.sleep(offTime)
    if btns.read() != 0:
        break

write_gpio(5, 0)
write_gpio(6, 0)
write_gpio(7, 0)
'''

#75% on, more bright compared to previous one
frequency = 100 #in Hz
dutyCycle = 0.75 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
oneCycleLen=1/frequency

onTime = dutyCycle*oneCycleLen
offTime = (1-dutyCycle)*oneCycleLen

print("frequency: %d Hz onTime: %f offTime: %f Sec oneCycleDuration:%f Sec" % (frequency,onTime,offTime,oneCycleLen))

gpioLedOnOff(onTime, offTime)
'''
while True:
    write_gpio(5, 1)
    write_gpio(6, 1)
    write_gpio(7, 1)
    time.sleep(onTime)
    write_gpio(5, 0)
    write_gpio(6, 0)
    write_gpio(7, 0)
    time.sleep(offTime)
    if btns.read() != 0:
        break

write_gpio(5, 0)
write_gpio(6, 0)
write_gpio(7, 0)
#'''

#time to find lowest possible frequency

# behavior is same as 100Hz, 50Hz i.e. no flickering observed
# Flickering observed with 45hz, 35Hz, 25Hz, 10Hz 

frequency = 50 #in Hz
dutyCycle = 0.75 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
oneCycleLen=1/frequency

onTime = dutyCycle*oneCycleLen
offTime = (1-dutyCycle)*oneCycleLen

gpioLedOnOff(onTime, offTime)
'''
while True:
    write_gpio(5, 1)
    write_gpio(6, 1)
    write_gpio(7, 1)
    time.sleep(onTime)
    write_gpio(5, 0)
    write_gpio(6, 0)
    write_gpio(7, 0)
    time.sleep(offTime)
    if btns.read() != 0:
        break

write_gpio(5, 0)
write_gpio(6, 0)
write_gpio(7, 0)
#'''

In [ ]:
import asyncio
cond = True
start= True
async def flash_leds():
    global cond, start
    print("white LED blinking every second")
    while cond:
        write_gpio(GREEN_PIN_ID, LED_ON)
        write_gpio(RED_PIN_ID, LED_ON)
        write_gpio(BLUE_PIN_ID, LED_ON)
        await asyncio.sleep(1)
        write_gpio(GREEN_PIN_ID, LED_OFF)
        write_gpio(RED_PIN_ID, LED_OFF)
        write_gpio(BLUE_PIN_ID, LED_OFF)
        await asyncio.sleep(1)

async def get_btns(_loop):
    global cond, start
    while start:
        await asyncio.sleep(0.01)
        if btns[0].read() != 0:
            #_loop.stop()
            cond = False
            #stopping everything & starting only GREEN
            write_gpio(GREEN_PIN_ID, LED_ON)

            write_gpio(RED_PIN_ID, LED_OFF)
            write_gpio(BLUE_PIN_ID, LED_OFF)
            #print("only GREEN LED is ON")
        if btns[1].read() != 0:
            #_loop.stop()
            cond = False
            #stopping everything & starting only RED
            write_gpio(RED_PIN_ID, LED_ON)

            write_gpio(GREEN_PIN_ID, LED_OFF)
            write_gpio(BLUE_PIN_ID, LED_OFF)
            #print("only RED LED is ON")
        if btns[2].read() != 0:
            #_loop.stop()
            cond = False
            #stopping everything & starting only blue
            write_gpio(BLUE_PIN_ID, LED_ON)

            write_gpio(GREEN_PIN_ID, LED_OFF)
            write_gpio(RED_PIN_ID, LED_OFF)
            #print("only BLUE LED is ON")
        if btns[3].read() != 0:
            #stopping everything & exit
            write_gpio(GREEN_PIN_ID, LED_OFF)
            write_gpio(RED_PIN_ID, LED_OFF)
            write_gpio(BLUE_PIN_ID, LED_OFF)
            _loop.stop() # exit both task
            cond = False
            start = False
            print("All Led Turned OFF")

loop = asyncio.new_event_loop()
loop.create_task(flash_leds())
loop.create_task(get_btns(loop))
loop.run_forever()
loop.close()
print("End of Program.")